# Bronze — Central Bank of Iceland Ingestion
Fetches the daily key interest rate, monthly CPI, and official exchange rates (EUR, USD) from the Central Bank XML API.

**Source:** Central Bank of Iceland XML time series API  
**Groups:** GroupID=1 (Interest rates), GroupID=3 (CPI), GroupID=9 (Official exchange rates)  
**Output:** `bronze.central_bank_raw` (Delta table)  
**Load mode:** `load_mode` = `incremental` (default) re-reads the last two months after the watermark and appends; `full` reloads from 1990 and overwrites. Silver keeps the latest version of each observation.


In [ ]:
# Parameters (set by the pipeline; toggle "parameter cell" in Fabric if it shows as a normal cell)
# "incremental": load only recent periods and append; "full": reload the whole history
load_mode = "incremental"


In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import timedelta
from pyspark.sql import functions as F
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Retry session for transient API errors
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])))
TIMEOUT_SECONDS = 120

BASE_URL = "https://sedlabanki.is/xmltimeseries/Default.aspx"
START_DATE_HISTORY = "1990-01-01"
LOOKBACK_DAYS = 62  # incremental: re-read the last two months, so late revisions and monthly averages are complete

if load_mode == "incremental" and spark.catalog.tableExists("bronze.central_bank_raw"):
    # Watermark: the latest observation already in Bronze
    last_date = (
        spark.table("bronze.central_bank_raw")
        .select(F.max(F.to_date("date_raw", "M/d/yyyy h:mm:ss a")))
        .first()[0]
    )
    start_date = (last_date - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d")
    is_full_load = False
    print(f"Incremental load: watermark {last_date}, fetching from {start_date}")
else:
    start_date = START_DATE_HISTORY
    is_full_load = True
    print(f"Full load: fetching history from {start_date}")


In [ ]:
def parse_central_bank_xml(xml_text):
    root = ET.fromstring(xml_text)
    records = []
    for ts in root.findall("TimeSeries"):
        ts_id, ts_name = ts.get("ID"), ts.find("Name").text
        for entry in ts.findall(".//Entry"):
            records.append({
                "group_name": root.find("Name").text,
                "series_id": ts_id,
                "series_name": ts_name,
                "date_raw": entry.find("Date").text,
                "value_raw": entry.find("Value").text
            })
    return pd.DataFrame(records)

In [ ]:
resp_rates = session.get(BASE_URL, params={"DagsFra": start_date, "GroupID": 1, "Type": "xml"}, timeout=TIMEOUT_SECONDS)
resp_rates.raise_for_status()
resp_cpi   = session.get(BASE_URL, params={"DagsFra": start_date, "GroupID": 3, "Type": "xml"}, timeout=TIMEOUT_SECONDS)
resp_cpi.raise_for_status()
resp_fx    = session.get(BASE_URL, params={"DagsFra": start_date, "GroupID": 9, "Type": "xml"}, timeout=TIMEOUT_SECONDS)
resp_fx.raise_for_status()

df_rates = parse_central_bank_xml(resp_rates.text)
df_cpi   = parse_central_bank_xml(resp_cpi.text)
df_fx    = parse_central_bank_xml(resp_fx.text)

new_data = pd.concat([df_rates, df_cpi, df_fx], ignore_index=True)
new_data["ingested_at"] = pd.Timestamp.now(tz="UTC")

spark_df = spark.createDataFrame(new_data)

if is_full_load:
    spark_df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable("bronze.central_bank_raw")
    print("Full history rebuild complete.")
else:
    spark_df.write.format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable("bronze.central_bank_raw")
    print(f"Appended {spark_df.count()} records.")